<a href="https://colab.research.google.com/github/pancakexia/machinelearning/blob/pancakexia_project/huggingface_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
!pip install ipdb
import ipdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.7 MB/s eta 0:00:00


In [26]:
!python -m ipdb your_script.py --your-arg value

<frozen runpy>:128: RuntimeWarning: 'ipdb.__main__' found in sys.modules after import of package 'ipdb', but prior to execution of 'ipdb.__main__'; this may result in unpredictable behaviour
Error: your_script.py does not exist


In [29]:
from huggingface_hub import notebook_login

notebook_login()

In [28]:
!pip install torch


In [31]:
!pip install -U transformers datasets evaluate accelerate timm

In [30]:
!pip install transformers datasets torch torchvision


In [32]:
!apt install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 34 not upgraded.


In [47]:
from datasets import load_dataset

# 加载 coco2017 数据集
dataset = load_dataset("phiyodr/coco2017", split="train")

# 打印前几条数据
print(dataset[0])


{'license': 3, 'file_name': 'train2017/000000391895.jpg', 'coco_url': 'http://images.cocodataset.org/train2017/000000391895.jpg', 'height': 360, 'width': 640, 'date_captured': '2013-11-14 11:18:45', 'flickr_url': 'http://farm9.staticflickr.com/8186/8119368305_4e622c8349_z.jpg', 'image_id': 391895, 'ids': [770337, 771687, 772707, 776154, 781998], 'captions': ['A man with a red helmet on a small moped on a dirt road. ', 'Man riding a motor bike on a dirt road on the countryside.', 'A man riding on the back of a motorcycle.', 'A dirt path with a young person on a motor bike rests to the foreground of a verdant area with a bridge and a background of cloud-wreathed mountains. ', 'A man in a red shirt and a red hat is on a motorcycle on a hill side.']}


In [49]:
# 图像预处理
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # 调整图像大小
    transforms.ToTensor(),          # 转换为 Tensor 格式
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 用ImageNet的均值和标准差进行归一化
])

# 选择一张图像进行测试
image_path = dataset[0]['coco_url']
image = Image.open(image_path)

# 应用预处理
input_image = transform(image).unsqueeze(0)  # 增加批次维度

FileNotFoundError: [Errno 2] No such file or directory: 'http://images.cocodataset.org/train2017/000000391895.jpg'

In [ ]:
!mkdir -p /content/coco2017
!wget -q http://images.cocodataset.org/zips/train2017.zip -O /content/coco2017/train2017.zip
!unzip -q /content/coco2017/train2017.zip -d /content/coco2017

In [ ]:
import os
from PIL import Image

coco_root = "/content/coco2017"
file_name = dataset[0]["file_name"]              # e.g. "train2017/000000391895.jpg"
image_path = os.path.join(coco_root, file_name)  # "/content/coco2017/train2017/000000391895.jpg"
image = Image.open(image_path).convert("RGB")


In [56]:
import os
from PIL import Image

# 1) 拿到相对路径
file_name = dataset[0]['file_name']
# → 'train2017/000000391895.jpg'

# 2) 本地根目录
coco_root = "/content/000000391895.jpg"

# 3) 拼出绝对路径
image_path = os.path.join(coco_root, file_name)
assert os.path.exists(image_path), f"{image_path} not found"

# 4) 打开并预处理
image = Image.open(image_path).convert("RGB")
input_image = transform(image).unsqueeze(0)

TypeError: join() argument must be str, bytes, or os.PathLike object, not 'list'

In [53]:
/mnt/data/coco2017/train2017/000000391895.jpg


SyntaxError: invalid imaginary literal (<ipython-input-53-ec3c749797ea>, line 1)

In [52]:
from transformers import ViTForImageClassification, ViTFeatureExtractor

# 加载预训练的 Vision Transformer 模型和特征提取器
model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224-in21k")
feature_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224-in21k")

# 通过特征提取器将图像处理为模型输入
inputs = feature_extractor(images=image, return_tensors="pt")


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


NameError: name 'image' is not defined

In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=1e-5)

# 假设你有标签数据
labels = torch.tensor([1])  # 示例标签，通常需要通过数据集加载实际标签

# 向前传播并计算损失
outputs = model(**inputs)
logits = outputs.logits  # 模型输出
loss = criterion(logits, labels)  # 计算损失

# 反向传播并更新模型参数
loss.backward()
optimizer.step()

print("Loss:", loss.item())



In [ ]:
from transformers import Trainer, TrainingArguments

# 设置训练参数
training_args = TrainingArguments(
    output_dir='./results',          # 输出目录
    num_train_epochs=3,              # 训练轮数
    per_device_train_batch_size=8,   # 每个设备的训练批次大小
    per_device_eval_batch_size=16,   # 每个设备的评估批次大小
    evaluation_strategy="epoch",     # 每个周期评估一次
    logging_dir='./logs',            # 日志目录
)

# 创建 Trainer
trainer = Trainer(
    model=model,                         # 预训练模型
    args=training_args,                  # 训练参数
    train_dataset=dataset,               # 训练数据集
    eval_dataset=dataset,                # 验证数据集
    compute_metrics=None,                # 计算评价指标（如果需要）
)

# 开始训练
trainer.train()


In [ ]:
results = trainer.evaluate()

print("Evaluation results:", results)


In [ ]:
# 加载测试图像
test_image = Image.open("test_image.jpg")
test_inputs = feature_extractor(test_image, return_tensors="pt")

# 使用训练后的模型进行预测
with torch.no_grad():
    outputs = model(**test_inputs)

# 获取预测结果
logits = outputs.logits
predicted_class = logits.argmax(-1).item()

print(f"Predicted class: {predicted_class}")
